In [2]:
# import pickle
# import joblib

In [3]:
# import pandas as pd
# from sklearn.model_selection import train_test_split
# from sklearn.pipeline import Pipeline
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.metrics import classification_report

In [4]:
from datasets import load_dataset
from transformers import CamembertTokenizer, CamembertForSequenceClassification, Trainer, TrainingArguments
from sklearn.preprocessing import LabelEncoder
import pandas as pd

c:\Users\maodo\Documents\Dev\Vie-publique\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# Chargement du CSV
df = pd.read_csv("C://Users//maodo//Documents//Dev//Vie-publique//questions-assemblee//datasets//resultats.csv")

In [6]:
# # Sauvegarder la rubrique et son occurrence
# value_count = df['rubrique'].value_counts()
# value_count_df = value_count.reset_index()
# value_count_df.columns = ['rubrique', 'occurence']
# value_count_df.to_csv("C://Users//maodo//Documents//Dev//Vie-publique//questions-assemblee//datasets//value_count.csv", index=False)

In [7]:
rubrique_mapping = {
    "sante": [
        "santé", "professions de santé", "établissements de santé", "maladies", "pharmacie", "médicaments",
        "assurance maladie", "sécurité sociale", "sang", "organes humains", "médecine", "fin de vie", "soins palliatifs"
    ],
    "education": [
        "enseignement", "enseignement supérieur", "formation professionnelle", "apprentissage", "enseignement secondaire",
        "enseignement maternel", "enseignement primaire", "enseignement privé", "enseignements artistiques", "enseignement agricole"
    ],
    "emploi": [
        "emploi", "chômage", "recherche d'emploi", "insertion professionnelle", "travail", "marché du travail", "contrat de travail", "conditions de travail", "droit du travail", "licenciement", "reconversion", "formation continue", "retraite anticipée"
    ],
    "fonction_publique": [
        "fonction publique", "fonctionnaires", "agents publics", "statut de la fonction publique", "recrutement public", "concours administratifs", "mutation", "carrière publique", "mobilité des agents", "retraites des fonctionnaires", "emplois publics", "administration centrale", "fonction publique hospitalière", "fonction publique territoriale"
    ],
    "securite": [
        "sécurité", "police", "défense", "sécurité routière", "terrorisme", "ordre public", "gendarmerie", "crimes", "délits", "contraventions", "sécurité des biens et des personnes", "sécurité civile", "pompiers", "protection civile"
    ],
    "agriculture": [
        "agriculture", "agriculteurs", "agriculture biologique", "agriculture durable", "agriculture urbaine", "agroalimentaire", "subventions agricoles", "politiques agricoles", "produits agricoles", "récoltes", "semences", "intrants agricoles", "coopératives agricoles"
    ],
    "elevage": [
        "élevage", "bétail", "animaux d'élevage", "aviculture", "porciculture", "élevage bovin", "élevage ovin", "élevage caprin", "production laitière", "production de viande", "santé animale", "bien-être animal"
    ],
    "peche": [
        "pêche", "pêche maritime", "pêche fluviale", "pêche artisanale", "pêche industrielle", "ressources halieutiques", "aquaculture", "pisciculture", "marins-pêcheurs", "gestion des stocks halieutiques", "quotas de pêche", "ports de pêche"
    ],
    "economie": [
        "économie", "finances", "impôts", "taxes", "commerce", "entreprises", "industrie", "banques", "pouvoir d'achat", "coût de la vie", "croissance économique", "investissement", "marché", "inflation", "PIB"
    ],
    "logement": [
        "logement", "habitat", "logement social", "urbanisme", "copropriété", "bâtiment", "travaux publics", "aides au logement", "prêts immobiliers", "accession à la propriété", "HLM", "rénovation urbaine"
    ],
    "environnement": [
        "environnement", "pollution", "biodiversité", "déchets", "climat", "catastrophes naturelles", "eau", "assainissement", "hydraulique",
        "cours d'eau", "étangs", "lacs", "mer", "littoral", "ruralité", "nuisances", "développement durable", "énergies renouvelables", "changement climatique", "protection de la nature", "forêts", "gestion de l'eau"
    ],
    "transport": [
        "transports", "routes", "infrastructures routières", "voirie", "automobiles", "cycles", "motocycles", "taxis", "transports routiers",
        "transports ferroviaires", "transports aériens", "transports par eau", "transports urbains", "mobilité", "trafic", "véhicules", "transports en commun"
    ],
    "culture": [
        "culture", "arts", "spectacles", "patrimoine", "musées", "cinéma", "théâtre", "musique", "danse", "peinture", "sculpture", "expositions", "festivals", "événements culturels", "création artistique", "patrimoine immatériel", "patrimoine matériel", "monuments historiques", "bibliothèques", "archives", "langue", "cérémonies"
    ],
    "presse": [
        "presse", "médias", "journalisme", "information", "audiovisuel", "radio", "télévision", "communication", "liberté de la presse", "agences de presse", "éditeurs", "livres", "édition", "publication", "diffusion", "internet d'information"
    ],
    "administration": [
        "administration", "services publics", "collectivités territoriales", "institutions sociales", "médico sociales", "chambres consulaires", "élus", "gouvernement", "ministères", "secrétariats d'État", "lois", "parlement", "état", "fonctionnement administratif", "réforme administrative"
    ],
    "fiscalite": [
        "impôts", "fiscalité", "impôts locaux", "impôt sur le revenu", "impôt sur les sociétés", "taxe", "douanes", "TVA", "contributions sociales", "régime fiscal", "contrôle fiscal"
    ],
    "relations-exterieures": [
        "coopération internationale", "affaires étrangères", "diplomatie", "ambassades", "consulats", "organisations internationales", "union européenne", "politique extérieure", "traités", "conventions", "relations bilatérales", "relations multilatérales", "aide au développement"
    ],
    "jeunesse": [
        "jeunesse", "jeunes", "étudiants", "lycéens", "collégiens", "enfants", "sports", "vie associative", "associations", "fondations", "engagement des jeunes", "volontariat", "service civique", "éducation populaire", "loisirs", "centres de loisirs", "scoutisme", "insertion des jeunes", "citoyenneté des jeunes"
    ],
    "numerique": [
        "numérique", "technologies", "télécommunications", "internet", "informatique", "cybersécurité", "réseaux", "digitalisation", "intelligence artificielle", "données", "open data", "innovation numérique", "start-up", "applications mobiles", "services en ligne", "économie numérique", "fracture numérique", "éducation au numérique"
    ],
    "justice": [
        "justice", "tribunal", "système judiciaire", "droits de l'homme", "professions judiciaires", "professions juridiques", "droit pénal", "discriminations", "harcèlement", "aide aux victimes", "avocats", "juges", "procureurs", "prison", "détention", "procès", "code pénal", "litige"
    ],
    "femme": [
        "femmes", "droits des femmes", "égalité femmes-hommes", "parité", "violences faites aux femmes", "santé des femmes", "condition féminine", "discrimination de genre", "égalité de genre", "lutte contre le sexisme"
    ],
    "famille_solidarite": [
        "famille", "solidarité", "aide sociale", "protection de l'enfance", "personnes âgées", "personnes handicapées", "allocations familiales", "prestations sociales", "aide aux familles", "aide aux personnes vulnérables", "inclusion sociale", "lutte contre la pauvreté", "soutien familial", "protection sociale", "aide à domicile"
    ],
    'energie': [
        'énergie', 'électricité', 'senelec', 'carburant', 'panne d\'électricité',
        'énergie solaire', 'pétrole', 'gaz', 'facture', 'centrale'
    ],
}

In [8]:
from rapidfuzz import fuzz, process

# On crée une liste de tuples (mot-clé, catégorie)
keywords = []
for cat, mots in rubrique_mapping.items():
    for mot in mots:
        keywords.append((mot, cat))

def simplifier_rubrique_fuzzy(rubrique, seuil=80):
    rubrique = rubrique.lower()
    # On cherche le mot-clé le plus proche
    best_match = process.extractOne(
        rubrique, [k[0] for k in keywords], scorer=fuzz.token_sort_ratio
    )
    if best_match and best_match[1] >= seuil:
        # On récupère la catégorie associée au mot-clé trouvé
        idx = [k[0] for k in keywords].index(best_match[0])
        return keywords[idx][1]
    return "autres"

# Fonction de normalisation
def simplifier_rubrique(rubrique):
    rubrique = rubrique.lower()
    for nouvelle, liste_anciennes in rubrique_mapping.items():
        for ancienne in liste_anciennes:
            if ancienne in rubrique:
                return nouvelle
    return simplifier_rubrique_fuzzy(rubrique)  # si aucune correspondance trouvée

In [9]:
# Application sur le DataFrame
df["thematique"] = df["rubrique"].apply(simplifier_rubrique)

# On renomme la colonne 'textQuestion' en 'question'
df = df.rename(columns={"textQuestion": "question"})

# Aperçu des nouvelles rubriques
df.head(10)

,uid,rubrique,question,thematique
0,QANR5L16QE1,animaux,M. Christophe Naegelen interroge M. le ministr...,autres
1,QANR5L16QE10,fonction publique hospitalière,Mme Marie-Pierre Rixain appelle l'attention de...,fonction_publique
2,QANR5L16QE100,associations et fondations,M. Pierre Cordier appelle l'attention de M. le...,jeunesse
3,QANR5L16QE1000,chômage,M. Didier Le Gac appelle l'attention de M. le ...,emploi
4,QANR5L16QE10000,santé,M. Olivier Falorni attire l'attention de M. le...,sante
5,QANR5L16QE10001,santé,M. Thomas Ménagé appelle l'attention de M. le ...,sante
6,QANR5L16QE10002,santé,M. Rodrigo Arenas alerte M. le ministre de la ...,sante
7,QANR5L16QE10003,santé,Mme Béatrice Descamps alerte M. le ministre de...,sante
8,QANR5L16QE10004,sécurité des biens et des personnes,Mme Isabelle Valentin appelle l'attention de M...,securite
9,QANR5L16QE10005,sécurité des biens et des personnes,Mme Sophie Mette interroge M. le ministre délé...,securite


In [10]:
df.tail(10)

,uid,rubrique,question,thematique
1990,QANR5L16QE1179,enseignement,Mme Caroline Colombier interroge M. le ministr...,education
1991,QANR5L16QE11790,fonction publique hospitalière,M. Florian Chauche interroge M. le ministre de...,fonction_publique
1992,QANR5L16QE11791,fonction publique territoriale,M. Christian Girard interroge M. le ministre d...,fonction_publique
1993,QANR5L16QE11792,fonctionnaires et agents publics,M. Daniel Labaronne appelle l'attention de M. ...,fonction_publique
1994,QANR5L16QE11793,fonctionnaires et agents publics,Mme Mathilde Panot appelle l'attention de M. l...,fonction_publique
1995,QANR5L16QE11794,fonctionnaires et agents publics,M. Julien Rancoule appelle l'attention de M. l...,fonction_publique
1996,QANR5L16QE11795,fonctionnaires et agents publics,M. Philippe Juvin appelle l'attention de M. le...,fonction_publique
1997,QANR5L16QE11796,formation professionnelle et apprentissage,M. Yannick Favennec-Bécot appelle l'attention ...,education
1998,QANR5L16QE11797,formation professionnelle et apprentissage,M. Stéphane Buchou appelle l'attention de Mme ...,education
1999,QANR5L16QE11798,Français de l'étranger,Mme Eléonore Caroit appelle l'attention de M. ...,autres


In [11]:
df['thematique'].value_counts()

thematique
autres                   291
sante                    222
education                195
environnement            171
securite                 147
economie                 116
energie                   95
administration            92
agriculture               79
fonction_publique         76
jeunesse                  71
logement                  69
transport                 69
justice                   64
famille_solidarite        54
relations-exterieures     40
elevage                   26
culture                   26
femme                     20
fiscalite                 19
emploi                    18
numerique                 14
presse                    13
peche                     13
Name: count, dtype: int64

In [12]:
df[df['thematique']=="autres"]["rubrique"].value_counts()

rubrique
animaux                                      45
communes                                     26
anciens combattants et victimes de guerre    21
consommation                                 19
retraites : généralités                      15
assurances                                   11
lieux de privation de liberté                11
élections et référendums                      7
drogue                                        7
prestations familiales                        7
assurance complémentaire                      6
produits dangereux                            6
immigration                                   6
gens du voyage                                6
laïcité                                       5
politique sociale                             5
Français de l'étranger                        5
donations et successions                      5
moyens de paiement                            4
professions et activités sociales             4
droits fondamentaux            

In [13]:
# Encodage des labels
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["thematique"])

# Enregistrement pour HuggingFace datasets
df[["question", "label"]].to_csv("C://Users//maodo//Documents//Dev//Vie-publique//questions-assemblee//datasets//train_data.csv", index=False)

In [14]:
# !pip install sentencepiece

In [15]:
from datasets import load_dataset

dataset = load_dataset("csv", data_files="C://Users//maodo//Documents//Dev//Vie-publique//questions-assemblee//datasets//train_data.csv")["train"]
dataset = dataset.train_test_split(test_size=0.2)

tokenizer = CamembertTokenizer.from_pretrained("camembert-base")

def tokenize(example):
    return tokenizer(example["question"], truncation=True, padding=True)

dataset = dataset.map(tokenize, batched=True)

model = CamembertForSequenceClassification.from_pretrained("camembert-base", num_labels=len(label_encoder.classes_))

Generating train split: 2000 examples [00:00, 26267.02 examples/s]
Map: 100%|██████████| 400/400 [00:00<00:00, 684.55 examples/s]
Some weights of CamembertForSequenceClassification were not initialized from the model checkpoint at camembert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [16]:
args = TrainingArguments(
    output_dir="C://Users//maodo//Documents//Dev//Vie-publique//questions-assemblee//datasets//results",
    eval_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=4,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
)

trainer.train()

c:\Users\maodo\Documents\Dev\Vie-publique\venv\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss
1,No log,2.099602
2,No log,1.621266
3,2.042300,1.440103
4,2.042300,1.395223


c:\Users\maodo\Documents\Dev\Vie-publique\venv\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\maodo\Documents\Dev\Vie-publique\venv\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=800, training_loss=1.6836410522460938, metrics={'train_runtime': 22804.7536, 'train_samples_per_second': 0.281, 'train_steps_per_second': 0.035, 'total_flos': 1684243375718400.0, 'train_loss': 1.6836410522460938, 'epoch': 4.0})

In [17]:
# Sauvegarde du modèle et de ses paramètres

# Sauvegarder le modèle entraîné
model.save_pretrained("C://Users//maodo//Documents//Dev//Vie-publique//questions-assemblee//datasets//camembert_model")

# Sauvegarder le tokenizer
tokenizer.save_pretrained("C://Users//maodo//Documents//Dev//Vie-publique//questions-assemblee//datasets//camembert_model")

# Sauvegarder le label encoder
import pickle
with open("C://Users//maodo//Documents//Dev//Vie-publique//questions-assemblee//datasets//label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

print("Modèle, tokenizer et label encoder sauvegardés avec succès.")


Modèle, tokenizer et label encoder sauvegardés avec succès.


In [18]:
def predict_question(question):
    inputs = tokenizer(question, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)
    predicted = outputs.logits.argmax().item()
    return label_encoder.inverse_transform([predicted])[0]


In [19]:
predict_question("Monsieur le ministre de la santé et de l'action sociale, vous avez dit récemment : \"Il y a trop de morts entre 19h et 6h dans les hôpitaux publics\". Cette situation alarmante de notre système de santé mérite une lecture fine pour des réponses urgentes afin de ne pas aboutir à une incrimination du personnel de santé de garde. \"trop de morts entre 19h et 6h dans les hôpitaux publics\" du Sénégal est surtout le résultat d'une mauvaise organisation structurelle et d'un manque flagrant de ressources humaines, dont le ministère de la santé que vous avez hérité porte une grande part de responsabilité. Monsieur le ministre, nos services de santé qui fonctionnent par exemple en journée avec cinq (05) médecins et cinq (05) à six (06) infirmiers se retrouvent la nuit avec un seul médecin de garde et un ou deux infirmiers. Une telle disproportion dans l'encadrement nuit forcément à la qualité des soins et met les patients en danger. Ce déficit d'effectifs est le reflet d'un manque de recrutement et d'anticipation des besoins en personnel qualifié. Le manque de personnel est tel que dans tous les services où vous allez la nuit, ce sont des médecins en cours de spécialisation qui font la garde, qui sont chirurgiens, qui sont gynécologues, qui sont anesthésistes réanimateurs, qui sont.... exploités sans revenu consistant, avec des volumes de travail importants. Monsieur le ministre, est-il prévu un recrutement d'urgence de personnel de santé ? Quelle mesure avez-vous prise pour améliorer les conditions des médecins en spécialisation, des internes et des médecins en général ? Monsieur le ministre, certains services d'urgence ou même des services spécialisés  sont confiés à des étudiants en 6e ou 7e année de médecine. Bien que ces braves étudiants soient souvent motivés et débrouillards, il est inacceptable de les charger seuls de la gestion de patients parfois complexes, sans encadrement suffisant. Un service d'urgence devrait être dirigé par un médecin urgentiste, ou à défaut, par un anesthésiste-réanimateur. Ces étudiants, qui n'ont pas encore achevé leur cursus, ne sont ni légalement ni techniquement préparés à porter cette responsabilité. Monsieur le ministre, quand allez vous prendre des mesures pour que les gardes dans nos structures de santé ne soient pas confiées qu'aux étudiants et aux stagiaires ? Monsieur le ministre, au manque d'expérience des étudiants et des stagiaires  s'ajoute le fait qu'ils se heurtent à l'indisponibilité des examens complémentaires et des moyens logistiques essentiels, surtout la nuit : pas d'ECG dans certains services d'urgence, absence de scopes de surveillance, pas de pousse-seringues électriques, ni même parfois de solutés ou de cathéters. Ce sont les familles des malades qui doivent sortir en pleine nuit acheter ces fournitures pour que le patient puisse être pris en charge. Avez-vous un plan d'équipement de nos structures de santé ?")

'sante'

In [21]:
# Charger le modèle, le tokenizer et le label encoder sauvegardés

from transformers import CamembertForSequenceClassification, CamembertTokenizer
import pickle

# Chemin vers le dossier où le modèle et le tokenizer ont été sauvegardés
model_dir = "C://Users//maodo//Documents//Dev//Vie-publique//questions-assemblee//datasets//camembert_model"

# Charger le modèle
model = CamembertForSequenceClassification.from_pretrained(model_dir)

# Charger le tokenizer
tokenizer = CamembertTokenizer.from_pretrained(model_dir)

# Charger le label encoder
with open("C://Users//maodo//Documents//Dev//Vie-publique//questions-assemblee//datasets//label_encoder.pkl", "rb") as f:
    label_encoder = pickle.load(f)

def predict_question2(question):
    """
    Prédit la thématique d'une question à partir du modèle sauvegardé.
    """
    inputs = tokenizer(question, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)
    predicted = outputs.logits.argmax().item()
    return label_encoder.inverse_transform([predicted])[0]


In [22]:
predict_question2("Monsieur le ministre de la santé et de l'action sociale, vous avez dit récemment : \"Il y a trop de morts entre 19h et 6h dans les hôpitaux publics\". Cette situation alarmante de notre système de santé mérite une lecture fine pour des réponses urgentes afin de ne pas aboutir à une incrimination du personnel de santé de garde. \"trop de morts entre 19h et 6h dans les hôpitaux publics\" du Sénégal est surtout le résultat d'une mauvaise organisation structurelle et d'un manque flagrant de ressources humaines, dont le ministère de la santé que vous avez hérité porte une grande part de responsabilité. Monsieur le ministre, nos services de santé qui fonctionnent par exemple en journée avec cinq (05) médecins et cinq (05) à six (06) infirmiers se retrouvent la nuit avec un seul médecin de garde et un ou deux infirmiers. Une telle disproportion dans l'encadrement nuit forcément à la qualité des soins et met les patients en danger. Ce déficit d'effectifs est le reflet d'un manque de recrutement et d'anticipation des besoins en personnel qualifié. Le manque de personnel est tel que dans tous les services où vous allez la nuit, ce sont des médecins en cours de spécialisation qui font la garde, qui sont chirurgiens, qui sont gynécologues, qui sont anesthésistes réanimateurs, qui sont.... exploités sans revenu consistant, avec des volumes de travail importants. Monsieur le ministre, est-il prévu un recrutement d'urgence de personnel de santé ? Quelle mesure avez-vous prise pour améliorer les conditions des médecins en spécialisation, des internes et des médecins en général ? Monsieur le ministre, certains services d'urgence ou même des services spécialisés  sont confiés à des étudiants en 6e ou 7e année de médecine. Bien que ces braves étudiants soient souvent motivés et débrouillards, il est inacceptable de les charger seuls de la gestion de patients parfois complexes, sans encadrement suffisant. Un service d'urgence devrait être dirigé par un médecin urgentiste, ou à défaut, par un anesthésiste-réanimateur. Ces étudiants, qui n'ont pas encore achevé leur cursus, ne sont ni légalement ni techniquement préparés à porter cette responsabilité. Monsieur le ministre, quand allez vous prendre des mesures pour que les gardes dans nos structures de santé ne soient pas confiées qu'aux étudiants et aux stagiaires ? Monsieur le ministre, au manque d'expérience des étudiants et des stagiaires  s'ajoute le fait qu'ils se heurtent à l'indisponibilité des examens complémentaires et des moyens logistiques essentiels, surtout la nuit : pas d'ECG dans certains services d'urgence, absence de scopes de surveillance, pas de pousse-seringues électriques, ni même parfois de solutés ou de cathéters. Ce sont les familles des malades qui doivent sortir en pleine nuit acheter ces fournitures pour que le patient puisse être pris en charge. Avez-vous un plan d'équipement de nos structures de santé ?")

'sante'